# Modelo de accesibilidad (Nearest facility)

Cálculo de tiempos de caminata hacia el vecino más cercano mediante Multi-source Dijkstra.

## Configuración de entorno

Importamos librerías y dependencias.

In [1]:
import geopandas as gpd
import osmnx as ox
import networkx as nx
from pathlib import Path

Configuración de OSMnx.

In [2]:
ox.settings.use_cache = True
ox.settings.log_console = False

Configuración de rutas.

In [3]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_FILE = PROCESSED_DIR / "lugares_datos_utiles_limpio.geojson"
GRAPH_FILE = PROCESSED_DIR / "grafo_rosario.graphml"

CALLES_FILE = PROCESSED_DIR / "calles.geojson"

## Carga de datos

In [4]:
gdf = gpd.read_file(PROCESSED_FILE)
grafo = ox.load_graphml(GRAPH_FILE)

In [5]:
nodos, calles = ox.graph_to_gdfs(grafo)

In [6]:
# Convertimos walk_time a número decimal
for u, v, k, data in grafo.edges(keys=True, data=True):
    if 'walk_time' in data:
        data['walk_time'] = float(data['walk_time'])

## Snapping

In [7]:
# Hacemos JOIN y lo guardamos en una variable temporal
gdf_snapped = gpd.sjoin_nearest(gdf, nodos, how="left")

# Eliminamos los empates espaciales
gdf_snapped = gdf_snapped[~gdf_snapped.index.duplicated(keep='first')]

gdf['nodo'] = gdf_snapped['osmid']

In [8]:
display(gdf[['name', 'categoria', 'nodo']].head())

,name,categoria,nodo
0,"Centro Municipal de Distrito Oeste ""Felipe Moré""",Comunidad,1314611725
1,"Centro Municipal de Distrito Sur ""Rosa Ziperov...",Comunidad,873819713
2,"Centro Municipal de Distrito Norte ""Villa Hort...",Comunidad,13293389611
3,Cementerio El Salvador,Servicios y Trámites,4852796672
4,Cementerio La Piedad,Servicios y Trámites,2680898414


## Multi-source Dijkstra

In [ ]:
categorias = gdf['categoria'].unique()

print("Inicializando Multi-source Dijkstra...")

for categoria in categorias:
    print(f"Procesando {categoria}...")
    
    # Agarramos todos los nodos donde hay un lugar de esa categoría
    nodos_origen = gdf[gdf['categoria'] == categoria]['nodo'].tolist()
    
    # Corremos el algoritmo: devuelve un diccionario {nodo_destino: tiempo_en_minutos}
    # Dijkstra inunda la red desde los orígenes y anota el camino más corto a cada nodo
    tiempos = nx.multi_source_dijkstra_path_length(grafo, nodos_origen, weight='walk_time')
    
    # Mapeamos ese diccionario a nuestro GeoDataFrame de esquinas
    # Si a un nodo no se puede llegar, le ponemos NaN
    nodos[f"time_{categoria.lower().replace(' ', '_')}"] = nodos.index.map(tiempos)

print("¡Cálculos finalizados!")

Inicializando Multi-source Dijkstra...
Procesando: Comunidad...
Procesando: Servicios y Trámites...
Procesando: Seguridad...
Procesando: Educación...
Procesando: Salud...
Procesando: Espacios Verdes...
Procesando: Espacios Culturales...
Procesando: Mercados y Comercios...
¡Cálculos finalizados!


In [10]:
# Filtramos buscando las columnas que acabamos de crear
columnas_time = [col for col in nodos.columns if col.startswith("time_")]

# Llenamos vacíos con 999 (inaccesible)
nodos[columnas_time] = nodos[columnas_time].fillna(999)

# Promediamos el tiempo entre el nodo origen (u) y destino (v) de cada calle
for col in columnas_time:
    calles[col] = (calles.index.get_level_values('u').map(nodos[col]) + 
                   calles.index.get_level_values('v').map(nodos[col])) / 2

# Calculamos cuáles cumplen la meta de los 15 minutos
cobertura_15m = calles[columnas_time] <= 15
calles['score_15min'] = cobertura_15m.sum(axis=1)

In [11]:
print(calles['score_15min'].value_counts().sort_index(ascending=False))

score_15min
8     2046
7     5144
6    12312
5    15824
4    12668
3     9938
2     4752
1     4146
0     2084
Name: count, dtype: int64


In [12]:
# Filtramos solo las columnas que nos importan para el mapa web
columnas = ['geometry', 'score_15min'] + columnas_time

# Exportamos el resultado
calles[columnas].copy().to_file(CALLES_FILE, driver='GeoJSON')
print(f"Archivo guardado exitosamente en: {CALLES_FILE.resolve()}")

Archivo guardado exitosamente en: D:\Documentos\MaxucaS\rosario-15-minute-city\data\processed\calles.geojson
